# Retail analytics exploration
Synthetic data. Read the data dictionary first.

In [1]:
from pathlib import Path
import pandas as pd
root = Path.cwd()
if not (root/'data').exists(): root=root.parent
df = pd.read_csv(root/'data/orders_clean.csv',parse_dates=['order_date'])
df.head()

,order_id,order_date,customer_id,region,product,category,unit_price,unit_cost,quantity,discount,shipping_cost,gross_sales,revenue,cost,profit,month,discount_band
0,O00302,2025-01-01,C2012,South,Pen Set,Stationery,250,110,1,0.0,70,250,250.0,180,70.0,2025-01,NaN
1,O00741,2025-01-01,C1145,South,Pen Set,Stationery,250,110,1,0.2,40,250,200.0,150,50.0,2025-01,High (>10%)
2,O00906,2025-01-01,C1228,East,Notebook,Stationery,150,65,4,0.2,70,600,480.0,330,150.0,2025-01,High (>10%)
3,O00984,2025-01-01,C1662,South,Notebook,Stationery,150,65,1,0.0,40,150,150.0,105,45.0,2025-01,NaN
4,O01006,2025-01-01,C1449,North,Backpack,Accessories,1800,850,2,0.2,150,3600,2880.0,1850,1030.0,2025-01,High (>10%)


In [2]:
df.groupby('month')[['revenue','profit']].sum()

,revenue,profit
month,,
2025-01,10178205.0,1654205.0
2025-02,8769985.0,1219805.0
2025-03,10893182.5,1647132.5
2025-04,11764732.5,1712117.5
2025-05,8709602.5,1438677.5
2025-06,10066540.0,1678445.0
2025-07,10752602.5,1624872.5
2025-08,10413720.0,1606015.0
2025-09,9388547.5,1407952.5


In [3]:
p=df.groupby('product')[['revenue','profit']].sum()
p['margin_pct']=100*p['profit']/p['revenue']
p.sort_values('profit')

,revenue,profit,margin_pct
product,,,
Notebook,239257.5,61682.5,25.780801
Pen Set,432412.5,151792.5,35.103634
Keyboard,3012840.0,1094640.0,36.332497
Backpack,3010230.0,1368460.0,45.460314
Headphones,4513875.0,1754775.0,38.875135
Office Chair,11062350.0,2914050.0,26.342052
Desk,15795450.0,3455850.0,21.878769
Laptop,82502500.0,7735120.0,9.375619


In [4]:
c=df.groupby('customer_id')['order_id'].nunique()
print('Repeat customer rate:',c.ge(2).mean())

Repeat customer rate: 0.755398854120758


Exercises: compare the discount and region outputs below with their SQL CSV files. Explain why discount comparisons are not causal evidence.

## Discount analysis

In [5]:
d=df.groupby('discount_band')[['revenue','profit']].sum()
d['margin_pct']=100*d['profit']/d['revenue']
d

,revenue,profit,margin_pct
discount_band,,,
High (>10%),25762840.0,-186095.0,-0.722339
Low (1-10%),53223675.0,9148855.0,17.189446


## Regional analysis

In [6]:
r=df.groupby('region').agg(revenue=('revenue','sum'),profit=('profit','sum'),orders=('order_id','count'))
r['margin_pct']=100*r['profit']/r['revenue']
r['aov']=r['revenue']/r['orders']
r.sort_values('profit',ascending=False)

,revenue,profit,orders,margin_pct,aov
region,,,,,
North,37427472.5,5707092.5,1781,15.248405,21014.863841
South,34724042.5,5532267.5,1766,15.932095,19662.538222
West,24574507.5,3866357.5,1287,15.733204,19094.411422
East,23755142.5,3412132.5,1156,14.363764,20549.431228
Unknown,87750.0,18520.0,10,21.105413,8775.000000
